## Part 2: Elliptic Curves over Finite Fields

In this section, we implement elliptic curve arithmetic over a finite field \(\mathbb{F}_p\), where \(p\) is a prime number. The curve has the same short Weierstrass form:

\[
E: y^2 = x^3 + ax + b \pmod p
\]

Unlike the real-number case, all operations are performed modulo \(p\). This means that every coordinate and coefficient is reduced using `% p`.

The point at infinity is represented by:

```python
(None, None)

In [12]:
class EllipticCurveFp:
    """
    Curva elíptica sobre el campo primo 𝔽_p.

    La curva sigue la forma de Weierstrass corta:
        E : y^2 = x^3 + a x + b (mod p)

    El punto al infinito se representa como (None, None).
    """

    def __init__(self, a: int, b: int, p: int) -> None:
        if p <= 2:
            raise ValueError("p must be greater than 2")

        self.p = p
        self.a = a % p
        self.b = b % p

        delta = (4 * self.a**3 + 27 * self.b**2) % p

        if delta == 0:
            raise ValueError("Curve is singular modulo p")

    def mod_inverse(self, k: int, p: int) -> int:
        k = k % p

        if k == 0:
            raise ZeroDivisionError("No inverse exists")

        a, b = k, p
        x0, x1 = 1, 0

        while b != 0:
            q = a // b

            a, b = b, a - q * b
            x0, x1 = x1, x0 - q * x1

        if a != 1:
            raise ZeroDivisionError("No inverse exists")

        return x0 % p

    def is_on_curve(self, P: tuple[int, int] | tuple[None, None]) -> bool:
        if P == (None, None):
            return True

        x, y = P

        left = (y * y) % self.p
        right = (x * x * x + self.a * x + self.b) % self.p

        return left == right

    def add_points(self, P: tuple, Q: tuple) -> tuple:
        if not self.is_on_curve(P):
            raise ValueError("Point not on curve")

        if not self.is_on_curve(Q):
            raise ValueError("Point not on curve")

        if P == (None, None):
            return Q

        if Q == (None, None):
            return P

        x1, y1 = P
        x2, y2 = Q
        p = self.p

        if x1 == x2 and (y1 + y2) % p == 0:
            return (None, None)

        if P == Q:
            m = ((3 * x1**2 + self.a) * self.mod_inverse(2 * y1, p)) % p
        else:
            m = ((y2 - y1) * self.mod_inverse(x2 - x1, p)) % p

        x3 = (m**2 - x1 - x2) % p
        y3 = (m * (x1 - x3) - y1) % p

        return (x3, y3)

    def scalar_multiply(self, k: int, P: tuple) -> tuple:
        if not self.is_on_curve(P):
            raise ValueError("Point not on curve")

        if k == 0 or P == (None, None):
            return (None, None)

        if k < 0:
            x, y = P
            P = (x, (-y) % self.p)
            k = -k

        result = (None, None)
        addend = P

        while k > 0:
            if k & 1:
                result = self.add_points(result, addend)

            addend = self.add_points(addend, addend)
            k >>= 1

        return result

In [11]:
def simulate_ecdh(curve: EllipticCurveFp, G: tuple, d_A: int, d_B: int) -> tuple:
    Q_A = curve.scalar_multiply(d_A, G)
    Q_B = curve.scalar_multiply(d_B, G)

    S_A = curve.scalar_multiply(d_A, Q_B)
    S_B = curve.scalar_multiply(d_B, Q_A)

    assert S_A == S_B
    return S_A

In [10]:
curve_fp = EllipticCurveFp(a=2, b=2, p=17)

assert curve_fp.is_on_curve((5, 1)) == True
assert curve_fp.add_points((5, 1), (5, 1)) == (6, 3)
assert curve_fp.add_points((5, 1), (10, 6)) == (3, 1)
assert curve_fp.scalar_multiply(10, (5, 1)) == (7, 11)
assert curve_fp.scalar_multiply(19, (5, 1)) == (None, None)

Q_A = curve_fp.scalar_multiply(3, (5, 1))
Q_B = curve_fp.scalar_multiply(7, (5, 1))

assert Q_A == (10, 6)
assert Q_B == (0, 6)

assert simulate_ecdh(curve_fp, (5, 1), 3, 7) == (6, 3)

print("Parte 2 OK: EllipticCurveFp y ECDH funcionan correctamente")

Parte 2 OK: EllipticCurveFp y ECDH funcionan correctamente
